# Hybrid LNN + Wave + Analogical Memory on Colab

This notebook trains the hybrid LNN with both imported NEXUS parts enabled on the full-xTB path:
- wave / quantum bridge
- analogical memory bank

Run top-to-bottom.

In [ ]:
# Cell 1: clone/pull repo and install package
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull --ff-only', shell=True, check=True)

subprocess.run(f'pip -q install -e {REPO_DIR}', shell=True, check=True)
print('Repo ready:', REPO_DIR)

In [ ]:
# Cell 2: mount Google Drive for persistent checkpoints/cache
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full', exist_ok=True)
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)

In [ ]:
# Cell 3: choose preset / overrides
import os

# Presets: fast, balanced, full
os.environ['HYBRID_COLAB_PRESET'] = 'balanced'

# Optional overrides
os.environ['HYBRID_COLAB_DATASET'] = 'data/prepared_training/main5_site_conservative_singlecyp_clean.json'
os.environ['HYBRID_COLAB_STRUCTURE_SDF'] = '3D structures.sdf'
os.environ['HYBRID_COLAB_OUTPUT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb'
os.environ['HYBRID_COLAB_ARTIFACT_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb'
os.environ['HYBRID_COLAB_MANUAL_CACHE_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full'
os.environ['HYBRID_COLAB_XTB_CACHE_DIR'] = '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb'

# hybrid+both defaults
os.environ['HYBRID_COLAB_FREEZE_NEXUS_MEMORY'] = '1'
os.environ['HYBRID_COLAB_SITE_LABELED_ONLY'] = '1'
os.environ['HYBRID_COLAB_COMPUTE_XTB_IF_MISSING'] = '0'

for key in [
    'HYBRID_COLAB_PRESET',
    'HYBRID_COLAB_DATASET',
    'HYBRID_COLAB_STRUCTURE_SDF',
    'HYBRID_COLAB_OUTPUT_DIR',
    'HYBRID_COLAB_FREEZE_NEXUS_MEMORY',
    'HYBRID_COLAB_SITE_LABELED_ONLY',
    'HYBRID_COLAB_COMPUTE_XTB_IF_MISSING',
]:
    print(f'{key}={os.environ[key]}')

In [ ]:
# Cell 4: train hybrid LNN + both imported NEXUS parts
exec(open('/content/enzyme_Software/scripts/colab_train_hybrid_lnn.py').read())

In [ ]:
# Cell 5: optional single-molecule smoke prediction after training
import os, subprocess
SMILES = 'CCOc1ccc2nc(S(N)(=O)=O)sc2c1'
CKPT = '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb/hybrid_full_xtb_best.pt'
if os.path.exists(CKPT):
    cmd = [
        'python', '/content/enzyme_Software/scripts/run_hybrid_prediction.py',
        '--smiles', SMILES,
        '--checkpoint', CKPT,
    ]
    subprocess.run(cmd, check=True)
else:
    print('Checkpoint not found yet:', CKPT)